# Machine Unlearning on ML-1M: All 4 Pipelines

This notebook runs **all four** unlearning pipelines sequentially on the same pretrained LightGCN checkpoint.

**Order (fastest → slowest):**
1. **AIE** – Attention-Based Influence Encoder (~2.5M params, 3 losses)
2. **CIE** – Causal Influence Encoder (~2.5M params, 5 losses)
3. **GAIE** – Graph Autoencoder Influence Encoder (VAE, 4 losses)
4. **HIE** – Hypernetwork-Based Influence Encoder (~40M params, 3 losses)

Each pipeline: **Unlearn → Fine-tune → Evaluate (Recall, NDCG, MI-BF, MI-NG)**

### Kaggle Setup
Enable **GPU accelerator** (Settings → Accelerator → GPU).

### Experimental protocol (aligned with UnlearnRec, SIGIR'25, Sec. 4.1.4)

**Threat model / unlearning target.** Adversarial edges are the least-probable user-item pairs
under a GCN trained on the clean data. The backbone LightGCN is trained **on the attacked graph**
(clean edges + injected adversarial edges), so the adversarial edges are genuinely learned as
positives. The unlearning task is to remove exactly those edges from the trained backbone.

**Why this matters.** If the backbone were trained on the clean graph instead, the adversarial
edges would never have been learned, the "before unlearning" scores would already be at or below
negative-sample level, and MI-BF / MI-NG would not measure forgetting. This notebook therefore
pretrains with `adversarial_attack=True`.

**Ground truth.** The exact-unlearning reference ("Retrain") is the same architecture retrained
from scratch on the residual graph (attacked graph minus adversarial edges = the clean data).

**Metrics.** MI-BF = mean recommendation probability of the unlearned edges before vs. after
unlearning (higher is better, must be > 1). MI-NG = mean probability of negative samples vs.
unlearned edges after unlearning (> 1 means unlearned edges are now less recommendable than
random non-edges). We also log the raw before/after/negative probabilities as a sanity check
that MI-NG is not trivially pre-satisfied.


---
## 0. Environment Setup

In [7]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

Installing torch-scatter + torch-sparse from: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 43.1 MB/s eta 0:00:00
torch_scatter version: 2.1.2+pt210cu128
torch_sparse version: 0.6.18+pt210cu128
setproctitle installed ✓


In [8]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

Repo already cloned.
Working directory: /kaggle/working/unlearnrec_improv
Contents: ['Utils', 'kaggle_pretrain_then_gaie_ml1m.ipynb', 'training', 'ckpt', 'README.md', 'datasets', 'unlearning', '.gitignore', 'logs', 'data', 'models', 'config', '.gitattributes', 'evaluation', 'state.db', 'examples', '.git']


In [9]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from Utils.utils import innerProduct, cal_mi_metrics, print_args
from models.Model import LightGCN, AIE, CIE, GAIE, HIE

print("All imports successful!")

All imports successful!


In [10]:
import torch as t
import numpy as np
import random
import time

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")
    print(f"Memory: {t.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


---
## 1. Pretrain LightGCN on ML-1M

Shared across all 4 pipelines. Only needs to run **once**.

In [ ]:
# ============================================================
# Hyperparameters for LightGCN pretraining on ML-1M
# ============================================================

args.data = 'ml1m'
args.model = 'lightgcn'
args.gpu = '0'
args.seed = 1234
args.lr = 1e-3
args.batch = 2048
args.epoch = 50              # GHOST: was 200
args.latdim = 128
args.gnn_layer = 3
args.reg = 1e-7
args.topk = 20
args.tst_epoch = 3          # GHOST: was 3
args.tst_bat = 256
args.decay = 1.0
args.bpr_wei = 1.0
# PAPER PROTOCOL (UnlearnRec Sec 4.1.4): the backbone to be unlearned must be
# trained on the ATTACKED graph so the adversarial edges are genuinely learned.
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'
args.save_path = './ckpt/pretrain_adv'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................2048
tst_bat......................................................................256
epoch........................................................................200
sim_epoch......................................................................5
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................................None
trained_model...............................................................None
save_path...................

In [6]:
handler = DataHandler()
handler.load_data(drop_rate=0.0, adv_attack=True)
# dropped_edges == the injected adversarial edges (the unlearning target)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges: {handler.trn_loader.dataset.__len__()}")
print(f"Test users: {handler.tst_loader.dataset.__len__()}")
print(f"Injected adversarial edges: {len(handler.adv_edges[0])}")

################here _load_one_file##################
################here _load_one_file##################


/kaggle/working/unlearnrec_improv/data/data_handler.py:59: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])
/kaggle/working/unlearnrec_improv/data/data_handler.py:67: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(col_degree, -0.5), [-1])
/kaggle/working/unlearnrec_improv/data/data_handler.py:77: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  return t.sparse.FloatTensor(idxs, vals, shape).cuda()


##############here in drop_rate <=0#################
Users: 6040, Items: 3706
Training edges: 720152
Test users: 6034


In [7]:
from training.pretrain_lightgcn import Coach as PretrainCoach

pretrain_coach = PretrainCoach(handler)
pretrain_coach.run()

print("\n" + "="*60)
print("Pretraining complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 9746
NUM OF EDGES 720152
2026-04-10 05:04:57.198970: Model Prepared
2026-04-10 05:04:57.199058: Model Initialized
2026-04-10 05:04:59.235261: Epoch 0/200, Topo: Recall = 0.0059, NDCG = 0.0111  


/kaggle/working/unlearnrec_improv/training/pretrain_lightgcn.py:134: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log('Step %d/%d: loss = %.6f, regLoss = %.6f         ' % (i, steps, loss, reg_loss), save=False, oneline=True)


2026-04-10 05:05:10.520966: Epoch 0/200, Trn: Loss = 0.3820, preLoss = 0.3818         
2026-04-10 05:05:11.918493: Epoch 0/200, Tst: Recall = 0.1169, NDCG = 0.1874   
2026-04-10 05:05:34.324897: Model Saved: ./ckpt/pretrain
2026-04-10 05:05:45.100575: Epoch 1/200, Trn: Loss = 0.3064, preLoss = 0.3059         
2026-04-10 05:05:55.741629: Epoch 2/200, Trn: Loss = 0.2722, preLoss = 0.2714         
2026-04-10 05:06:06.611636: Epoch 3/200, Trn: Loss = 0.2509, preLoss = 0.2498         
2026-04-10 05:06:07.975184: Epoch 3/200, Tst: Recall = 0.1665, NDCG = 0.2590   
2026-04-10 05:06:30.187214: Model Saved: ./ckpt/pretrain
2026-04-10 05:06:41.164616: Epoch 4/200, Trn: Loss = 0.2368, preLoss = 0.2355         
2026-04-10 05:06:52.102896: Epoch 5/200, Trn: Loss = 0.2271, preLoss = 0.2255         
2026-04-10 05:07:03.244916: Epoch 6/200, Trn: Loss = 0.2175, preLoss = 0.2157         
2026-04-10 05:07:04.611912: Epoch 6/200, Tst: Recall = 0.1917, NDCG = 0.2899   
2026-04-10 05:07:26.721068: Model Sav

In [ ]:

# Load the best checkpoint (saved when Recall improved during training)
PRETRAINED_PATH = args.save_path
ckp = t.load(PRETRAINED_PATH + '.mod', weights_only=False)
pretrained_model = ckp['model'].cuda()
pretrained_model.eval()

reses = pretrain_coach.tst_epoch(pretrained_model)
print(f"\nBest Pretrained LightGCN Performance:")
print(f"  Recall@{args.topk}: {reses['Recall']:.4f}")
print(f"  NDCG@{args.topk}:   {reses['NDCG']:.4f}")


2026-04-10 05:47:41.260387: Steps 23/23: recall = 30.56, ndcg = 38.00          
Pretrained LightGCN Performance:
  Recall@20: 0.1950
  NDCG@20:   0.2644


---
# Results Collection

We'll store results from each pipeline for a final comparison.

In [22]:
# Dictionary to collect results from all pipelines
all_results = {}

---
## 1b. Retrain reference (exact-unlearning ground truth)

The paper's "Retrain" row: the same backbone retrained from scratch on the residual graph
(= clean training data). Used as ground truth for both utility (Recall/NDCG) and efficacy
(MI-BF/MI-NG), and as the efficiency yardstick (unlearning must beat this wall-clock time).

In [ ]:
# ============================================================
# Retrain reference (exact unlearning): train from scratch on E_r
# ============================================================
RUN_RETRAIN_REFERENCE = True   # set False to skip and save GPU time

if RUN_RETRAIN_REFERENCE:
    args.adversarial_attack = False
    args.save_path = './ckpt/retrain_ref'
    handler_clean = DataHandler()
    handler_clean.load_data(drop_rate=0.0, adv_attack=False)

    retrain_coach = PretrainCoach(handler_clean)
    t.cuda.reset_peak_memory_stats()
    _t0 = time.time()
    retrain_coach.run()
    retrain_time = time.time() - _t0
    retrain_mem = t.cuda.max_memory_allocated() / 1e9

    ckp_r = t.load('./ckpt/retrain_ref.mod', weights_only=False)
    retrain_model = ckp_r['model'].cuda()
    retrain_model.eval()
    retrain_metrics = retrain_coach.tst_epoch(retrain_model)
    print(f"\n[Retrain] Recall@{args.topk}: {retrain_metrics['Recall']:.4f}, "
          f"NDCG@{args.topk}: {retrain_metrics['NDCG']:.4f}, "
          f"time: {retrain_time:.1f}s, peak mem: {retrain_mem:.2f} GB")


In [ ]:
if RUN_RETRAIN_REFERENCE:
    # MI metrics of the retrained model on the adversarial edges.
    # "Before" scores come from the attacked backbone (the model being unlearned).
    args.adversarial_attack = True
    handler_adv_eval = DataHandler()
    handler_adv_eval.load_data(drop_rate=0.0, adv_attack=True)
    adv_u, adv_i = handler_adv_eval.dropped_edges

    with t.no_grad():
        ret_u_emb, ret_i_emb = retrain_model.forward(handler_clean.ts_ori_adj, keepRate=1.0)
        atk_u_emb, atk_i_emb = pretrained_model.forward(handler_adv_eval.ts_ori_adj, keepRate=1.0)

    ret_drp = innerProduct(ret_u_emb[adv_u], ret_i_emb[adv_i])
    atk_drp = innerProduct(atk_u_emb[adv_u], atk_i_emb[adv_i])

    rows, cols = handler_adv_eval.ori_trn_mat.row, handler_adv_eval.ori_trn_mat.col
    edge_set = set(zip(rows.tolist(), cols.tolist()))
    neg_r, neg_c = [], []
    while len(neg_r) < len(adv_u):
        i, j = np.random.randint(args.user), np.random.randint(args.item)
        if (i, j) not in edge_set:
            edge_set.add((i, j)); neg_r.append(i); neg_c.append(j)
    ret_neg = innerProduct(ret_u_emb[neg_r], ret_i_emb[neg_c])

    retrain_mi = cal_mi_metrics(ret_drp, ret_neg, before_drp_scores=atk_drp)
    all_results['Retrain'] = {
        'Recall': retrain_metrics['Recall'], 'NDCG': retrain_metrics['NDCG'],
        'MI_BF': retrain_mi['mi_bf'], 'MI_NG': retrain_mi['mi_ng'],
        'BeforeProb': retrain_mi['avg_before_prob'], 'AfterProb': retrain_mi['avg_after_prob'],
        'NegProb': retrain_mi['avg_neg_prob'],
        'UnlearnTime': retrain_time, 'FinetuneTime': 0.0, 'PeakMemGB': retrain_mem,
    }
    print(f"[Retrain] MI-BF={retrain_mi['mi_bf']:.4f}, MI-NG={retrain_mi['mi_ng']:.4f}")

    del retrain_coach, handler_clean
    t.cuda.empty_cache()


---
---
# Pipeline 1: AIE (Attention-Based Influence Encoder)

**Fastest pipeline** — ~2.5M params, 3 losses (BPR + unlearn + alignment).

Deleted edges → GAT over influence graph → MLP shift generator → ΔE

**No reconstruction loss** (unlike GAIE).

## AIE — Step 2a: Unlearn

In [ ]:
# ============================================================
# Hyperparameters for AIE unlearning on ML-1M
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30              # GHOST: was 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.3
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/aie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................................None
trained_model...................................................../ckpt/pretrain
save_path...................

In [11]:
handler_unlearn_aie = DataHandler()
handler_unlearn_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_aie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_aie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_aie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################


/kaggle/working/unlearnrec_improv/data/data_handler.py:59: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])
/kaggle/working/unlearnrec_improv/data/data_handler.py:67: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(col_degree, -0.5), [-1])


##############here in drop_rate >0#################
Users: 6040, Items: 3706
Training edges (after drop): 720406
Dropped edges: 9746
Picked (retained) edges: 720406


In [12]:
from unlearning.aie_unlearn import Coach as AIECoach

aie_coach = AIECoach(handler_unlearn_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_coach.run()
aie_unlearn_time = time.time() - _t0
aie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE unlearn] {aie_unlearn_time:.1f}s, peak GPU mem {aie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 05:47:49.508551: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 05:47:49.508577: Model Initialized
2026-04-10 05:47:51.377787: Epoch 0/30, Topo: Recall = 0.2270, NDCG = 0.3222   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.227027,  NDCG: 0.322232 @ Epoch: 0= 33.14, ndcg = 43.96          
[AIE] AIE test:
  Dropped edges  (mean,var,max,min): <-4.3246, 2.4036, 2.5952, -11.7086>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <5.8521, 2.8542, 15.5079, -5.2543>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <0.9782, 6.4731, 11.8867, -7.8671>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.3028, MI-AUC=0.0346, MI-ACC=0.9868
  MI pretrai

## AIE — Step 2b: Fine-Tune

In [ ]:
args.model_2_finetune = './ckpt/aie_unlearn'

args.fineTune = True
args.epoch = 20              # increased from 15 for better recovery with learnable shift MLP
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.1
args.align_wei = 0.05
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/aie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune.............................................../ckpt/aie_unlearn
trained_model...................................................../ckpt/pretrain
save_path...................

In [14]:
handler_ft_aie = DataHandler()
handler_ft_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

aie_ft_coach = AIECoach(handler_ft_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_ft_coach.run()
aie_ft_time = time.time() - _t0
aie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE finetune] {aie_ft_time:.1f}s, peak GPU mem {aie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 06:07:13.680934: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 06:07:19.022551: Model Loaded
2026-04-10 06:07:20.787847: Epoch 0/15, Topo: Recall = 0.2270, NDCG = 0.3222   
2026-04-10 06:07:37.183459: Epoch -4/15, Trn: Loss = 0.1439, preLoss = 0.1171, unlearn_loss = 0.0405, align_loss = 1.2570   
2026-04-10 06:07:53.882966: Epoch -3/15, Trn: Loss = 0.1375, preLoss = 0.1172, unlearn_loss = 0.0315, align_loss = 0.7866   
2026-04-10 06:08:10.361747: Epoch -2/15, Trn: Loss = 0.1351, preLoss = 0.1168, unlearn_loss = 0.0285, align_loss = 0.6470   
2026-04-10 06:08:26.940859: Epoch -1/15, Trn: Loss = 0.1336, preLoss = 0.1163, unlearn_loss = 0.0263, align_loss = 0.595

## AIE — Step 3: Evaluate

In [15]:
handler_eval_aie = DataHandler()
handler_eval_aie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    aie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned AIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

aie_ft_coach.handler = handler_eval_aie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned AIE model from ./ckpt/aie_ft.mod


In [16]:
aie_metrics = aie_ft_coach.tst_epoch(aie_ft_coach.model)
print(f"\n[AIE] Recall@{args.topk}: {aie_metrics['Recall']:.4f}")
print(f"[AIE] NDCG@{args.topk}:   {aie_metrics['NDCG']:.4f}")

2026-04-10 06:18:36.962389: Steps 23/23: recall = 32.63, ndcg = 43.27          
[AIE] Recall@20: 0.2239
[AIE] NDCG@20:   0.3165


In [ ]:
aie_mi = aie_ft_coach.test_unlearn(aie_ft_coach.model, prefix='[AIE] Final Evaluation')

[AIE] [AIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-4.4822, 1.7016, 1.3438, -12.3933>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <5.5803, 2.5258, 11.7508, -5.1060>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <0.8836, 6.0739, 11.1389, -7.6709>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.3658, MI-AUC=0.0217, MI-ACC=0.9868
  MI pretrain: MI-BF=0.0260, MI-NG=3.2394, MI-AUC=0.1365, MI-ACC=0.9868


5.365825533866882

In [ ]:
all_results['AIE'] = {
    'Recall': aie_metrics['Recall'], 'NDCG': aie_metrics['NDCG'],
    'MI_BF': aie_mi['mi_bf'], 'MI_NG': aie_mi['mi_ng'],
    'BeforeProb': aie_mi['avg_before_prob'], 'AfterProb': aie_mi['avg_after_prob'],
    'NegProb': aie_mi['avg_neg_prob'],
    'UnlearnTime': aie_unlearn_time, 'FinetuneTime': aie_ft_time,
    'PeakMemGB': max(aie_unlearn_mem, aie_ft_mem),
}
print("AIE results stored.")

# Free GPU memory
del aie_coach, aie_ft_coach, handler_unlearn_aie, handler_ft_aie, handler_eval_aie
t.cuda.empty_cache()
print("AIE objects freed, GPU cache cleared.")

NameError: name 'aie_metrics' is not defined

---
---
# Pipeline 2: CIE (Causal Influence Encoder)

**Second fastest** — ~2.5M params, 5 losses (BPR + unlearn + alignment + contrastive + causal).

Models unlearning as a **causal intervention** do(e_ij = 0).
Computes **counterfactual embeddings** E^cf once, then trains with extra consistency losses.

CIE-specific params: `contrast_wei=0.01`, `causal_wei=0.1`.

## CIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for CIE unlearning on ML-1M
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30              # GHOST: was 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.2
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.contrast_wei = 0.01
args.causal_wei = 0.15
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/cie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune.............................................../ckpt/aie_unlearn
trained_model...................................................../ckpt/pretrain
save_path...................

In [23]:
handler_unlearn_cie = DataHandler()
handler_unlearn_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_cie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_cie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_cie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 6040, Items: 3706
Training edges (after drop): 720406
Dropped edges: 9746
Picked (retained) edges: 720406


In [24]:
from unlearning.cie_unlearn import Coach as CIECoach

cie_coach = CIECoach(handler_unlearn_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_coach.run()
cie_unlearn_time = time.time() - _t0
cie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE unlearn] {cie_unlearn_time:.1f}s, peak GPU mem {cie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 06:24:34.283982: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 06:24:34.284007: Model Initialized
2026-04-10 06:24:35.975902: Epoch 0/30, Topo: Recall = 0.2241, NDCG = 0.3237   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.224085,  NDCG: 0.323675 @ Epoch: 0= 32.30, ndcg = 43.92          
[CIE] CIE test:
  Dropped edges  (mean,var,max,min): <-3.5022, 3.9720, 9.2041, -11.3543>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <7.7989, 4.2645, 16.4079, -4.6783>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <1.7918, 8.4557, 15.2715, -7.5835>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.2941, MI-AUC=0.0635, MI-ACC=0.9868
  MI pretrai

## CIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/cie_unlearn'

args.fineTune = True
args.epoch = 15              # GHOST: was 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.contrast_wei = 0.005
args.causal_wei = 0.15
args.perf_degrade = 0.5
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/cie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune.............................................../ckpt/cie_unlearn
trained_model...................................................../ckpt/pretrain
save_path...................

In [26]:
handler_ft_cie = DataHandler()
handler_ft_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

cie_ft_coach = CIECoach(handler_ft_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_ft_coach.run()
cie_ft_time = time.time() - _t0
cie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE finetune] {cie_ft_time:.1f}s, peak GPU mem {cie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 07:06:04.675070: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 07:06:10.346564: Model Loaded
2026-04-10 07:06:12.022504: Epoch 0/15, Topo: Recall = 0.1176, NDCG = 0.1831   
2026-04-10 07:06:32.454874: Epoch -4/15, Trn: Loss = 27.6619, preLoss = 0.3811, unlearn_loss = 0.1695, align_loss = 2717.2268, contrast_loss = 0.3246, causal_loss = 0.6526  
2026-04-10 07:06:52.741498: Epoch -3/15, Trn: Loss = 0.6156, preLoss = 0.1745, unlearn_loss = 0.0231, align_loss = 40.1907, contrast_loss = 0.1365, causal_loss = 0.2705  
2026-04-10 07:07:13.254301: Epoch -2/15, Trn: Loss = 0.4258, preLoss = 0.1516, unlearn_loss = 0.0189, align_loss = 24.2694, contrast_loss = 0.0972, ca

## CIE — Step 3: Evaluate

In [27]:
handler_eval_cie = DataHandler()
handler_eval_cie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    cie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned CIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

cie_ft_coach.handler = handler_eval_cie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned CIE model from ./ckpt/cie_ft.mod


In [28]:
cie_metrics = cie_ft_coach.tst_epoch(cie_ft_coach.model)
print(f"\n[CIE] Recall@{args.topk}: {cie_metrics['Recall']:.4f}")
print(f"[CIE] NDCG@{args.topk}:   {cie_metrics['NDCG']:.4f}")

2026-04-10 07:28:35.631186: Steps 23/23: recall = 27.96, ndcg = 32.40          
[CIE] Recall@20: 0.1864
[CIE] NDCG@20:   0.2498


In [ ]:
cie_mi = cie_ft_coach.test_unlearn(cie_ft_coach.model, prefix='[CIE] Final Evaluation')

[CIE] [CIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-5.8664, 29.5688, 41.4255, -37.8711>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <8.5468, 8.8132, 23.8693, -13.0912>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <2.7778, 9.4383, 39.9668, -41.3483>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=8.6442, MI-AUC=0.0609, MI-ACC=0.9868
  MI pretrain: MI-BF=0.0260, MI-NG=3.2394, MI-AUC=0.1365, MI-ACC=0.9868


8.644246578216553

In [ ]:
all_results['CIE'] = {
    'Recall': cie_metrics['Recall'], 'NDCG': cie_metrics['NDCG'],
    'MI_BF': cie_mi['mi_bf'], 'MI_NG': cie_mi['mi_ng'],
    'BeforeProb': cie_mi['avg_before_prob'], 'AfterProb': cie_mi['avg_after_prob'],
    'NegProb': cie_mi['avg_neg_prob'],
    'UnlearnTime': cie_unlearn_time, 'FinetuneTime': cie_ft_time,
    'PeakMemGB': max(cie_unlearn_mem, cie_ft_mem),
}
print("CIE results stored.")

del cie_coach, cie_ft_coach, handler_unlearn_cie, handler_ft_cie, handler_eval_cie
t.cuda.empty_cache()
print("CIE objects freed, GPU cache cleared.")

CIE results stored.
CIE objects freed, GPU cache cleared.


---
---
# Pipeline 3: GAIE (Graph Autoencoder Influence Encoder)

**Third fastest** — VAE-based, 4 losses (BPR + unlearn + alignment + reconstruction).

Influence graph → VAE encoder → latent z → reparameterize → shift MLP → ΔE

GAIE-specific: `rec_wei=0.1` (reconstruction loss).

## GAIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for GAIE unlearning on ML-1M
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30              # GHOST: was 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.5
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gaie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


NameError: name 'PRETRAINED_PATH' is not defined

In [32]:
handler_unlearn_gaie = DataHandler()
handler_unlearn_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_gaie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_gaie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_gaie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 6040, Items: 3706
Training edges (after drop): 720406
Dropped edges: 9746
Picked (retained) edges: 720406


In [33]:
from unlearning.gaie_unlearn import Coach as GAIECoach

gaie_coach = GAIECoach(handler_unlearn_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_coach.run()
gaie_unlearn_time = time.time() - _t0
gaie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE unlearn] {gaie_unlearn_time:.1f}s, peak GPU mem {gaie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 07:29:07.399982: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 07:29:07.400006: Model Initialized
2026-04-10 07:29:09.042761: Epoch 0/30, Topo: Recall = 0.0286, NDCG = 0.0543  
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.027763,  NDCG: 0.052527 @ Epoch: 0= 3.96, ndcg = 7.19           
[GAIE] GAIE test:
  Dropped edges  (mean,var,max,min): <-0.0439, 188.9975, 95.8195, -56.1218>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <17.2846, 208.8051, 79.2585, -42.9538>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <5.2846, 182.6369, 82.6876, -68.3973>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.3285, MI-AUC=0.3882, MI-ACC=0.9868
  

## GAIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/gaie_unlearn'

args.fineTune = True
args.epoch = 15              # GHOST: was 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gaie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................../ckpt/gaie_unlearn
trained_model...................................................../ckpt/pretrain
save_path...................

In [35]:
handler_ft_gaie = DataHandler()
handler_ft_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

gaie_ft_coach = GAIECoach(handler_ft_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_ft_coach.run()
gaie_ft_time = time.time() - _t0
gaie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE finetune] {gaie_ft_time:.1f}s, peak GPU mem {gaie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 08:02:16.896474: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 08:02:22.323488: Model Loaded
2026-04-10 08:02:23.938240: Epoch 0/15, Topo: Recall = 0.0311, NDCG = 0.0402  
2026-04-10 08:02:42.388886: Epoch -4/15, Trn: Loss = 6.0041, preLoss = 0.2316, unlearn_loss = 0.2063, align_loss = 556.0618, rec_loss = 1.6454   
2026-04-10 08:03:00.633489: Epoch -3/15, Trn: Loss = 0.2626, preLoss = 0.1223, unlearn_loss = 0.0536, align_loss = 4.9469, rec_loss = 0.7400  
2026-04-10 08:03:18.784176: Epoch -2/15, Trn: Loss = 0.2389, preLoss = 0.1203, unlearn_loss = 0.0490, align_loss = 3.1479, rec_loss = 0.7121  
2026-04-10 08:03:37.270497: Epoch -1/15, Trn: Loss = 0.2240, pre

## GAIE — Step 3: Evaluate

In [ ]:
handler_eval_gaie = DataHandler()
handler_eval_gaie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    gaie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned GAIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

gaie_ft_coach.handler = handler_eval_gaie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned GAIE model from ./ckpt/gaie_ft.mod


In [ ]:
gaie_metrics = gaie_ft_coach.tst_epoch(gaie_ft_coach.model)
print(f"\n[GAIE] Recall@{args.topk}: {gaie_metrics['Recall']:.4f}")
print(f"[GAIE] NDCG@{args.topk}:   {gaie_metrics['NDCG']:.4f}")

2026-04-10 08:20:26.657784: Steps 23/23: recall = 33.06, ndcg = 43.93          
[GAIE] Recall@20: 0.2254
[GAIE] NDCG@20:   0.3206


In [ ]:
gaie_mi = gaie_ft_coach.test_unlearn(gaie_ft_coach.model, prefix='[GAIE] Final Evaluation')

[GAIE] [GAIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-4.1140, 2.2940, 4.2123, -12.1554>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <5.7812, 2.6354, 12.2406, -4.4043>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <0.9878, 6.2329, 11.6905, -9.1680>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.1018, MI-AUC=0.0377, MI-ACC=0.9868
  MI pretrain: MI-BF=0.0260, MI-NG=3.2394, MI-AUC=0.1365, MI-ACC=0.9868


5.101816833019257

In [ ]:
all_results['GAIE'] = {
    'Recall': gaie_metrics['Recall'], 'NDCG': gaie_metrics['NDCG'],
    'MI_BF': gaie_mi['mi_bf'], 'MI_NG': gaie_mi['mi_ng'],
    'BeforeProb': gaie_mi['avg_before_prob'], 'AfterProb': gaie_mi['avg_after_prob'],
    'NegProb': gaie_mi['avg_neg_prob'],
    'UnlearnTime': gaie_unlearn_time, 'FinetuneTime': gaie_ft_time,
    'PeakMemGB': max(gaie_unlearn_mem, gaie_ft_mem),
}
print("GAIE results stored.")

del gaie_coach, gaie_ft_coach, handler_unlearn_gaie, handler_ft_gaie, handler_eval_gaie
t.cuda.empty_cache()
print("GAIE objects freed, GPU cache cleared.")

GAIE results stored.
GAIE objects freed, GPU cache cleared.


---
---
# Pipeline 4: HIE (Hypernetwork-Based Influence Encoder)

**Slowest pipeline** — ~40M params from HyperNetwork, 3 losses (BPR + unlearn + alignment).

Influence graph → GNN → mean-pool → latent z → HyperNetwork H(z) → W_u → ΔE = W_u ⊙ E

The HyperNetwork's `fc_row: Linear(128, 9746×32)` ≈ 40M params makes this significantly heavier.

## HIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for HIE unlearning on ML-1M
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30             # GHOST: was 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/hie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................................None
trained_model..................../kaggle/working/unlearnrec_improv/ckpt/pretrain
save_path...................

In [14]:
handler_unlearn_hie = DataHandler()
handler_unlearn_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_hie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_hie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_hie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 6040, Items: 3706
Training edges (after drop): 720406
Dropped edges: 9746
Picked (retained) edges: 720406


In [15]:
from unlearning.hie_unlearn import Coach as HIECoach

hie_coach = HIECoach(handler_unlearn_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_coach.run()
hie_unlearn_time = time.time() - _t0
hie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE unlearn] {hie_unlearn_time:.1f}s, peak GPU mem {hie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 14:37:22.156457: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 14:37:22.156555: Model Initialized
2026-04-10 14:37:24.076633: Epoch 0/30, Topo: Recall = 0.2270, NDCG = 0.3220   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################


/kaggle/working/unlearnrec_improv/data/data_handler.py:59: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])
/kaggle/working/unlearnrec_improv/data/data_handler.py:67: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(col_degree, -0.5), [-1])


##############here in drop_rate >0#################
>>>>Recall: 0.227027,  NDCG: 0.322047 @ Epoch: 0= 33.31, ndcg = 43.92          
[HIE] HIE test:
  Dropped edges  (mean,var,max,min): <-4.3236, 2.4061, 2.5827, -11.6706>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <5.8489, 2.8539, 15.4807, -5.2622>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <0.9771, 6.4704, 11.8840, -7.8516>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.3007, MI-AUC=0.0347, MI-ACC=0.9868
  MI pretrain: MI-BF=0.0260, MI-NG=3.2394, MI-AUC=0.1365, MI-ACC=0.9868
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
2026-04-10 14:38:15.654395: Model Saved: ./ckpt/hie_unlearn


/kaggle/working/unlearnrec_improv/unlearning/hie_unlearn.py:148: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  log('Step %d/%d: loss = %.6f, regLoss = %.6f, unlearn = %.6f, align = %.6f         '


2026-04-10 14:38:51.107971: Epoch 0/30, Trn: Loss = 2.3948, preLoss = 0.1837, unlearn_loss = 3.7057, align_loss = 7.0427    
2026-04-10 14:38:52.747979: Epoch 0/30, Tst: Recall = 0.2048, NDCG = 0.2735    
2026-04-10 14:39:27.934347: Epoch 1/30, Trn: Loss = 2.1781, preLoss = 0.1950, unlearn_loss = 3.1278, align_loss = 8.2624     
2026-04-10 14:40:02.947607: Epoch 2/30, Trn: Loss = 2.0758, preLoss = 0.2019, unlearn_loss = 2.9007, align_loss = 8.3484    
2026-04-10 14:40:38.033287: Epoch 3/30, Trn: Loss = 2.0064, preLoss = 0.2075, unlearn_loss = 2.7546, align_loss = 8.3086    
2026-04-10 14:41:13.171213: Epoch 4/30, Trn: Loss = 1.9557, preLoss = 0.2117, unlearn_loss = 2.6507, align_loss = 8.2490    
2026-04-10 14:41:48.528171: Epoch 5/30, Trn: Loss = 1.9165, preLoss = 0.2153, unlearn_loss = 2.5682, align_loss = 8.2206    
2026-04-10 14:41:50.183091: Epoch 5/30, Tst: Recall = 0.1950, NDCG = 0.2709    
2026-04-10 14:42:25.209360: Epoch 6/30, Trn: Loss = 1.8841, preLoss = 0.2181, unlearn_los

## HIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/hie_unlearn'

args.fineTune = True
args.epoch = 15              # GHOST: was 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10           # GHOST: was 10
args.tst_epoch = 5           # GHOST: was 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/hie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune.............................................../ckpt/hie_unlearn
trained_model..................../kaggle/working/unlearnrec_improv/ckpt/pretrain
save_path...................

In [17]:
handler_ft_hie = DataHandler()
handler_ft_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

hie_ft_coach = HIECoach(handler_ft_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_ft_coach.run()
hie_ft_time = time.time() - _t0
hie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE finetune] {hie_ft_time:.1f}s, peak GPU mem {hie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 9746
NUM OF EDGES 720406
2026-04-10 14:57:57.737959: Model Preparedecall = 33.12, ndcg = 43.73          
2026-04-10 14:58:03.174994: Model Loaded
2026-04-10 14:58:04.879957: Epoch 0/15, Topo: Recall = 0.1893, NDCG = 0.2818   
2026-04-10 14:58:22.498844: Epoch -4/15, Trn: Loss = 0.1497, preLoss = 0.1194, unlearn_loss = 0.0409, align_loss = 1.5982   
2026-04-10 14:58:40.180296: Epoch -3/15, Trn: Loss = 0.1375, preLoss = 0.1174, unlearn_loss = 0.0315, align_loss = 0.7622   
2026-04-10 14:58:57.893359: Epoch -2/15, Trn: Loss = 0.1350, preLoss = 0.1167, unlearn_loss = 0.0276, align_loss = 0.6686   
2026-04-10 14:59:16.070477: Epoch -1/15, Trn: Loss = 0.1335, preLoss = 0.1160, unlearn_loss = 0.0248, align_loss = 0.644

## HIE — Step 3: Evaluate

In [18]:
handler_eval_hie = DataHandler()
handler_eval_hie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    hie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned HIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

hie_ft_coach.handler = handler_eval_hie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned HIE model from ./ckpt/hie_ft.mod


In [19]:
hie_metrics = hie_ft_coach.tst_epoch(hie_ft_coach.model)
print(f"\n[HIE] Recall@{args.topk}: {hie_metrics['Recall']:.4f}")
print(f"[HIE] NDCG@{args.topk}:   {hie_metrics['NDCG']:.4f}")

2026-04-10 15:09:52.894106: Steps 23/23: recall = 32.99, ndcg = 43.95          
[HIE] Recall@20: 0.2250
[HIE] NDCG@20:   0.3171


In [ ]:
hie_mi = hie_ft_coach.test_unlearn(hie_ft_coach.model, prefix='[HIE] Final Evaluation')

[HIE] [HIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-4.5240, 1.6930, 2.4299, -11.4912>  |  Pretrain: <-2.2624,3.7153,8.6342,-8.6002>
  Positive edges (mean,var,max,min): <5.6281, 2.5294, 11.6711, -4.9659>  |  Pretrain: <5.5665,2.6796,11.8157,-12.4488>
  Negative edges (mean,var,max,min): <0.8938, 6.0962, 11.1995, -8.0384>  |  Pretrain: <0.9770,5.5869,11.3501,-14.4595>
  MI metrics:  MI-BF=0.0260, MI-NG=5.4178, MI-AUC=0.0214, MI-ACC=0.9868
  MI pretrain: MI-BF=0.0260, MI-NG=3.2394, MI-AUC=0.1365, MI-ACC=0.9868


5.41777116060257

In [ ]:
all_results['HIE'] = {
    'Recall': hie_metrics['Recall'], 'NDCG': hie_metrics['NDCG'],
    'MI_BF': hie_mi['mi_bf'], 'MI_NG': hie_mi['mi_ng'],
    'BeforeProb': hie_mi['avg_before_prob'], 'AfterProb': hie_mi['avg_after_prob'],
    'NegProb': hie_mi['avg_neg_prob'],
    'UnlearnTime': hie_unlearn_time, 'FinetuneTime': hie_ft_time,
    'PeakMemGB': max(hie_unlearn_mem, hie_ft_mem),
}
print("HIE results stored.")

del hie_coach, hie_ft_coach, handler_unlearn_hie, handler_ft_hie, handler_eval_hie
t.cuda.empty_cache()
print("HIE objects freed, GPU cache cleared.")

HIE results stored.
HIE objects freed, GPU cache cleared.


---
---
# Final Comparison: All 4 Pipelines

In [ ]:
import json

print("\n" + "#" * 110)
print("#" + " FINAL COMPARISON — attacked-backbone protocol (UnlearnRec Sec. 4.1.4) ".center(108) + "#")
print("#" * 110)
print(f"#  Dataset:  {args.data} ({args.user} users, {args.item} items)")
print(f"#  Backbone: LightGCN ({args.latdim}-dim, {args.gnn_layer} layers), trained on attacked graph (adv method: {args.adv_method})")
print(f"#  Unlearn target: all injected adversarial edges")
print("#" + "-" * 108 + "#")
header = (f"#  {'Method':<9} {'Recall@20':>10} {'NDCG@20':>9} {'MI-BF':>8} {'MI-NG':>8} "
          f"{'P(before)':>10} {'P(after)':>9} {'P(neg)':>8} {'Time(s)':>9} {'Mem(GB)':>8}  #")
print(header)
print("#" + "-" * 108 + "#")

for name in ['Retrain', 'AIE', 'CIE', 'GAIE', 'HIE']:
    r = all_results.get(name, {})
    def f(key, fmt='{:.4f}'):
        return fmt.format(r[key]) if key in r else 'N/A'
    total_time = (r.get('UnlearnTime', 0) or 0) + (r.get('FinetuneTime', 0) or 0)
    time_s = f"{total_time:.1f}" if 'UnlearnTime' in r else 'N/A'
    print(f"#  {name:<9} {f('Recall'):>10} {f('NDCG'):>9} {f('MI_BF'):>8} {f('MI_NG'):>8} "
          f"{f('BeforeProb'):>10} {f('AfterProb'):>9} {f('NegProb'):>8} {time_s:>9} {f('PeakMemGB', '{:.2f}'):>8}  #")

print("#" + "-" * 108 + "#")
print("#  Sanity check: P(before) should sit near positive-edge level (edges were trained on);         #")
print("#  a method truly forgets when P(after) <= P(neg). Speedup = Retrain time / method time.        #")
print("#" * 110)

os.makedirs('./logs', exist_ok=True)
out = {
    'dataset': args.data, 'backbone': 'lightgcn', 'latdim': args.latdim,
    'gnn_layer': args.gnn_layer, 'seed': args.seed, 'adv_method': args.adv_method,
    'protocol': 'attacked-backbone (UnlearnRec Sec 4.1.4)', 'results': all_results,
}
res_file = f'./logs/final_results_{args.data}.json'
with open(res_file, 'w') as fs:
    json.dump(out, fs, indent=2)
print(f"\nResults saved to {res_file} — download this from Kaggle output for the paper.")



######################################################################
#                  ALL PIPELINES — FINAL COMPARISON                  #
######################################################################
#  Dataset:    ML-1M (6040 users, 3706 items)
#  Backbone:   LightGCN (128-dim, 3 layers)
#  Drop Mode:  Adversarial (lightgcn0.5)
#  Drop Rate:  10% of training edges
#--------------------------------------------------------------------#
#  Pipeline      Recall@20      NDCG@20     Losses      ~Params  #
#--------------------------------------------------------------------#
#  AIE                 N/A          N/A          3        ~2.5M  #
#  CIE                 N/A          N/A          5        ~2.5M  #
#  GAIE                N/A          N/A          4          ~3M  #
#  HIE              0.2250       0.3171          3         ~40M  #
#--------------------------------------------------------------------#
#  (MI-BF and MI-NG printed above by each pipeline's test_unlearn)
###